# 02 — Silver Layer
Clean the data, fix bad values, and join the two files together.

## Load the raw data

In [1]:
import pandas as pd
import numpy as np

tickets   = pd.read_csv("it_support_tickets.csv", parse_dates=["created_at"])
customers = pd.read_csv("customer_info_1000.csv")

## Mask sensitive columns
The customer file contains real personal data (names, emails, addresses). We hash these columns using SHA-256 before doing anything else so we never work with raw PII.

In [2]:
import hashlib

def hash_value(val):
    if pd.isna(val):
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()[:16]

# Columns containing personal information
pii_columns = ["customer_name", "contact_person", "email_address", "Address_line_1"]

for col in pii_columns:
    if col in customers.columns:
        customers[col] = customers[col].apply(hash_value)

print("PII columns masked:")
customers[pii_columns].head()


PII columns masked:


,customer_name,contact_person,email_address,Address_line_1
0,01332c876518a793,NaN,c4d25e9c90ff23e9,2e58196c7c240eca
1,6cea57c2fb6cbc2a,NaN,836f82db99121b34,948595bd0490ff95
2,NaN,db8db1c8e00fd626,4a084e90c098ac98,d8c1c4659640b640
3,NaN,3e5f2809db48e9c4,4c2b54287c2fa921,65c8bea6303617b8
4,NaN,6cea57c2fb6cbc2a,a41cd188c7ed3023,8b2e7fba8625e03a


## Join customer info onto tickets

In [3]:
# Bring in country and city from the customer file
df = tickets.merge(customers[["customer_id", "country", "city"]], on="customer_id", how="left")

print("Rows after join:", len(df))
df[["ticket_id", "customer_id", "country", "city", "region"]].head()

Rows after join: 100711


,ticket_id,customer_id,country,city,region
0,TCKT_000001,CUST_00861,Germany,Berlin,EU
1,TCKT_000002,CUST_00770,United States,New York,NaN
2,TCKT_000003,CUST_02559,United Arab Emirates,Abu Dhabi,MEA
3,TCKT_000004,CUST_03557,Brazil,São Paulo,LATAM
4,TCKT_000005,CUST_09556,United States,Beverly Hills,NaN


## Fix missing region values
About 20% of tickets are missing a region. We infer it from the customer's country.

In [4]:
country_to_region = {
    "United States": "NA",  "Canada": "NA",    "Mexico": "NA",
    "Germany": "EU",        "France": "EU",    "United Kingdom": "EU",
    "Spain": "EU",          "Italy": "EU",     "Netherlands": "EU",
    "Sweden": "EU",         "Poland": "EU",    "Belgium": "EU",
    "China": "APAC",        "Japan": "APAC",   "India": "APAC",
    "Australia": "APAC",    "South Korea": "APAC", "Singapore": "APAC",
    "Brazil": "LATAM",      "Argentina": "LATAM",  "Colombia": "LATAM",
    "Chile": "LATAM",       "Peru": "LATAM",
    "South Africa": "MEA",  "Nigeria": "MEA",  "Kenya": "MEA",
    "Saudi Arabia": "MEA",  "UAE": "MEA",      "Egypt": "MEA",
}

missing = df["region"].isna()
print("Missing region before:", missing.sum())

df.loc[missing, "region"] = df.loc[missing, "country"].map(country_to_region)

print("Missing region after: ", df["region"].isna().sum())
df = df.dropna(subset=["region"])
print("Final row count:", len(df))

Missing region before: 20134
Missing region after:  18005


Final row count: 82706


## Fix CSAT score
A score of 0 means the customer didn't fill in the survey — it's not a real score. We recode it to null.

In [5]:
print("Before:", df["csat_score"].value_counts().sort_index().to_dict())
df["csat_score"] = df["csat_score"].replace(0, np.nan)
print("After: ", df["csat_score"].value_counts(dropna=False).sort_index().to_dict())

Before: {0: 24757, 1: 4823, 2: 12322, 3: 16888, 4: 14315, 5: 9601}
After:  {1.0: 4823, 2.0: 12322, 3.0: 16888, 4.0: 14315, 5.0: 9601, nan: 24757}


## Add useful columns

In [6]:
# Year-month for time series charts
df["year_month"] = df["created_at"].dt.to_period("M").astype(str)

# Simple boolean: is the ticket resolved or closed?
df["is_resolved"] = df["status"].isin(["resolved", "closed_no_action"])

print("Sample of new columns:")
df[["ticket_id", "year_month", "is_resolved"]].head()

Sample of new columns:


,ticket_id,year_month,is_resolved
0,TCKT_000001,2024-01,True
1,TCKT_000002,2024-10,True
2,TCKT_000003,2024-06,False
3,TCKT_000004,2025-12,False
4,TCKT_000005,2023-08,True


## Final check

In [7]:
print("Total rows:", len(df))
print()
print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Total rows: 82706

Remaining nulls:
resolution_summary       32983
resolution_time_hours    32983
csat_score               24757
country                  70876
city                     70876
dtype: int64
